In [1]:
!pip install google-ads

Defaulting to user installation because normal site-packages is not writeable


In [2]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
import json
from datetime import datetime, timedelta
from google.ads.googleads.client import GoogleAdsClient
from google.oauth2 import service_account
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_batch

In [4]:
CONFIG = {
    "developer_token": "UfwukMYOFwQLFsCXVjjtzw",
    "login_customer_id": "6385295998", 
    "customer_id": "1849790507", 
    "service_account_file": "/home/hadoop/agrim_cdp/base_tables/google_ads/base/agrim-tech-servic-account-key.json", 
    "scopes": ["https://www.googleapis.com/auth/adwords"],
}


In [5]:
def get_service_account_credentials():
    """Creates credentials from service account JSON file."""
    try:
        credentials = service_account.Credentials.from_service_account_file(
            CONFIG["service_account_file"],
            scopes=CONFIG["scopes"]
        )
        return credentials
    except FileNotFoundError:
        print(f"❌ Service account file '{CONFIG['service_account_file']}' not found!")
        print("Please download your service account JSON file from Google Cloud Console")
        return None
    except Exception as e:
        print(f"❌ Error loading service account: {e}")
        return None

In [6]:
def get_google_ads_client():
    """Initializes Google Ads client with service account credentials."""
    credentials = get_service_account_credentials()
    if not credentials:
        return None
    
    # Create client configuration
    client_config = {
        "developer_token": CONFIG["developer_token"],
        "login_customer_id": CONFIG["login_customer_id"],
        "use_proto_plus": True,
    }
    
    # Initialize client with service account credentials
    client = GoogleAdsClient(
        credentials=credentials,
        developer_token=CONFIG["developer_token"],
        login_customer_id=CONFIG["login_customer_id"],
        use_proto_plus=True
    )
    
    return client

In [7]:
def test_google_ads_connection():
    """Tests if the API connection works with service account."""
    try:
        client = get_google_ads_client()
        if not client:
            return False
            
        service = client.get_service("GoogleAdsService")
        
        query = "SELECT customer.id, customer.descriptive_name FROM customer"
        request = client.get_type("SearchGoogleAdsRequest")
        request.customer_id = CONFIG["customer_id"]
        request.query = query
        
        response = service.search(request=request)
        for row in response:
            print(f"✅ Success! Connected to account: {row.customer.descriptive_name} (ID: {row.customer.id})")
            return True
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nPossible fixes:")
        print("- Ensure service account has Google Ads API access")
        print("- Check if service account is linked to your Google Ads account")
        print("- Verify the service account JSON file is correct")
        return False

In [8]:
def get_campaigns():
    """Example function to fetch campaigns."""
    try:
        client = get_google_ads_client()
        if not client:
            return []
            
        service = client.get_service("GoogleAdsService")
        
        query = """
            SELECT 
                  campaign.id,
                  campaign.name,
                  campaign.status,
                  campaign.advertising_channel_type,
                  metrics.impressions,
                  metrics.clicks,
                  metrics.cost_micros
            FROM campaign
            WHERE campaign.status != 'REMOVED'
            AND segments.date DURING LAST_7_DAYS
        """
        
        request = client.get_type("SearchGoogleAdsRequest")
        request.customer_id = CONFIG["customer_id"]
        request.query = query
        
        response = service.search(request=request)
        campaigns = []
        
        for row in response:
            campaign_data = (
                row.campaign.id,
                row.campaign.name,
                row.campaign.status.name,
                row.campaign.advertising_channel_type.name,
                row.metrics.impressions,  # Corrected to metrics
                row.metrics.clicks,  # Corrected to metrics
                row.metrics.cost_micros,  # Corrected to metrics
                row.metrics.cost_micros / 1_000_000,
                round((row.metrics.clicks / row.metrics.impressions * 100), 2) if row.metrics.impressions > 0 else 0,
                round(((row.metrics.cost_micros / 1_000_000) / row.metrics.clicks), 2) if row.metrics.clicks > 0 else 0
            )
            campaigns.append(campaign_data)
            #print(f"📊 Campaign: {campaign_data['name']} (ID: {campaign_data['id']}) - Status: {campaign_data['status']}")
        
        return campaigns
        
    except Exception as e:
        print(f"❌ Error fetching campaigns: {e}")
        return []

In [9]:
def get_account_performance(days_back=7):
    """Gets account performance metrics for the last N days."""
    try:
        client = get_google_ads_client()
        if not client:
            return None
            
        service = client.get_service("GoogleAdsService")
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        
        query = f"""
            SELECT 
                metrics.impressions,
                metrics.clicks,
                metrics.cost_micros,
                metrics.conversions,
                segments.date
            FROM customer
            WHERE segments.date >= '{start_date.strftime('%Y-%m-%d')}'
            AND segments.date <= '{end_date.strftime('%Y-%m-%d')}'
        """
        
        request = client.get_type("SearchGoogleAdsRequest")
        request.customer_id = CONFIG["customer_id"]
        request.query = query
        
        response = service.search(request=request)
        
        total_impressions = 0
        total_clicks = 0
        total_cost = 0
        total_conversions = 0
        
        for row in response:
            total_impressions += row.metrics.impressions
            total_clicks += row.metrics.clicks
            total_cost += row.metrics.cost_micros
            total_conversions += row.metrics.conversions
        
        # Convert cost from micros to currency
        total_cost_currency = total_cost / 1_000_000
        
        performance = {
            "date_range": f"{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}",
            "impressions": total_impressions,
            "clicks": total_clicks,
            "cost": round(total_cost_currency, 2),
            "conversions": total_conversions,
            "ctr": round((total_clicks / total_impressions * 100), 2) if total_impressions > 0 else 0,
            "cpc": round((total_cost_currency / total_clicks), 2) if total_clicks > 0 else 0
        }
        
        print(f"\n📈 Account Performance ({performance['date_range']}):")
        print(f"   Impressions: {performance['impressions']:,}")
        print(f"   Clicks: {performance['clicks']:,}")
        print(f"   Cost: ${performance['cost']:,}")
        print(f"   Conversions: {performance['conversions']}")
        print(f"   CTR: {performance['ctr']}%")
        print(f"   CPC: ${performance['cpc']}")
        
        return performance
        
    except Exception as e:
        print(f"❌ Error fetching performance data: {e}")
        return None

In [10]:
if __name__ == "__main__":
    print("🔍 Testing Google Ads API connection with Service Account...")
    
    if test_google_ads_connection():
        print("\n📋 Fetching campaigns...")
        campaigns = get_campaigns()
        
        print(f"\n📊 Found {len(campaigns)} campaigns")

        upsert_query = sql.SQL("""
                                INSERT INTO cdp_raw_db.google_campaign_stats (
                                    campaign_id, campaign_name, campaign_status, channel_type, impressions, clicks, cost_micros, cost, click_thr, cost_per_click
                                ) 
                                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                                ON CONFLICT (campaign_id) 
                                DO UPDATE SET
                                    campaign_name = EXCLUDED.campaign_name,
                                    campaign_status = EXCLUDED.campaign_status,
                                    channel_type = EXCLUDED.channel_type,
                                    impressions = EXCLUDED.impressions,
                                    clicks = EXCLUDED.clicks,
                                    cost_micros = EXCLUDED.cost_micros,
                                    cost = EXCLUDED.cost,
                                    click_thr = EXCLUDED.click_thr,
                                    cost_per_click = EXCLUDED.cost_per_click;
                            """)

        connection = get_rds_connection()
        cursor = connection.cursor()

        # Use execute_batch for bulk insert/update
        execute_batch(cursor, upsert_query, campaigns)
        
        # Commit the transaction to make changes persistent
        connection.commit()
        
        # Close the cursor and connection
        cursor.close()
        connection.close()

        print("\n📈 Load Into RDS Completed Successfully")
        
        print("\n📈 Fetching account performance...")
        performance = get_account_performance(days_back=7)
    else:
        print("\n❌ Connection failed. Please check your service account setup.")

🔍 Testing Google Ads API connection with Service Account...


✅ Success! Connected to account: Agrim (ID: 1849790507)

📋 Fetching campaigns...



📊 Found 114 campaigns

📈 Load Into RDS Completed Successfully

📈 Fetching account performance...



📈 Account Performance (2025-07-05 to 2025-07-12):
   Impressions: 20,828,923
   Clicks: 330,509
   Cost: $1,082,145.01
   Conversions: 44085.0
   CTR: 1.59%
   CPC: $3.27
